# 📖 Notebook 4: End-to-End Encryption Basics

Welcome to the **final notebook** in our WhatsApp System Design series! 🎉

In the previous notebooks, we explored message delivery, read receipts, and group messaging.
Now we tackle one of the most important features of modern messaging: **End-to-End Encryption (E2E)**.

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. **Understand** what End-to-End Encryption is and why it matters
2. **Learn** how public and private keys work together
3. **Encrypt and decrypt** messages using RSA (a real encryption algorithm)
4. **See** why the server (and its admins!) cannot read your messages
5. **Explore** how group encryption works and its challenges

---

## 📚 Series Overview

| # | Notebook | Status |
|---|---------|--------|
| 1 | Message Delivery & Storage | ✅ |
| 2 | Read Receipts & Presence | ✅ |
| 3 | Group Messaging | ✅ |
| 4 | **End-to-End Encryption Basics** | 👈 You are here |

> 💡 **Good news**: Most of this notebook works with just Python and the `cryptography` library.
> We only need PostgreSQL to store and retrieve keys/messages — no WebSocket server required!

---

## 🛠️ Setup

### Prerequisites

**1. Start the Docker containers** (if not already running):

```bash
cd 06-system-designs/whatsapp   # from the repo root
docker compose up -d postgres adminer
```

> We only need `postgres` and `adminer` for this notebook — no Redis or WebSocket server needed!

**2. Select the correct Jupyter kernel:**

- In VS Code, click the kernel picker (top-right of the notebook)
- Select the `.venv` kernel from this project
- If the kernel doesn't appear, reload the VS Code window (`Cmd+Shift+P` → "Reload Window")

**3. Verify Adminer** (optional but handy):

- Open http://localhost:8080
- Login: System=PostgreSQL, Server=postgres, User=demo, Password=demo, Database=whatsapp_demo

In [ ]:
# === 📦 Imports ===
# Standard library
import json
import base64

# Database
import psycopg2
from psycopg2.extras import RealDictCursor

# Cryptography - this is our star library for this notebook!
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization

print("✅ All imports successful!")
print("🔐 We're ready to learn about encryption!")

In [ ]:
# === 🔌 Database Connection Helper ===

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "whatsapp_demo",
    "user": "demo",
    "password": "demo",
}

def get_db_connection():
    """Create a new database connection."""
    return psycopg2.connect(**DB_CONFIG)

def run_query(query, params=None, fetch=True):
    """Run a SQL query and optionally return results as dictionaries."""
    conn = get_db_connection()
    conn.autocommit = True
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cur:
            cur.execute(query, params)
            if fetch:
                return cur.fetchall()
            return None
    finally:
        conn.close()

# Quick test
users = run_query("SELECT id, username, display_name FROM users ORDER BY id;")
print("✅ Database connected! Found users:")
for u in users:
    print(f"   👤 {u['username']} ({u['display_name']})")

---

## 📬 What is End-to-End Encryption?

Let's start with a simple analogy that makes this concept click.

### The Postal Analogy 📮

Imagine you want to send a secret message to your friend across town.

**Without End-to-End Encryption** — it's like sending a **postcard**:

```
📝 "Hey Bob, the surprise party is at 7pm!"

You write it → Mailman can read it → Post office can read it → Bob reads it
```

Everyone who handles the postcard can read your message. The post office, the mailman,
even a nosy neighbor could peek at it. **No privacy at all!**

---

**With End-to-End Encryption** — it's like sending a **locked box** 📦🔒:

```
📦 You put the message in a box and lock it with Bob's padlock.

You lock it → Mailman carries it (can't open it!) → Post office stores it (can't open it!) → Bob unlocks it with his key
```

Only Bob has the key to his padlock. Nobody in between can read the message —
not the mailman, not the post office, not even the company that made the box!

---

### How It Works in WhatsApp 💬

```
┌───────┐                    ┌──────────┐                    ┌───────┐
│ Alice │                    │  Server  │                    │  Bob  │
│       │   🔒 encrypted    │          │   🔒 encrypted    │       │
│ "Hi!" │ ──────────────►   │ #@$!%&*  │ ──────────────►   │ "Hi!" │
│       │   message          │ (can't   │   message          │       │
│       │                    │  read!)  │                    │       │
└───────┘                    └──────────┘                    └───────┘
  encrypts                    just stores                     decrypts
  with Bob's                  & forwards                      with his
  public key                  the blob                        private key
```

The server is like the post office — it **delivers** messages but **cannot read** them.

### What WhatsApp Actually Uses

WhatsApp uses the **Signal Protocol**, which is very sophisticated. It includes:
- Double Ratchet Algorithm (for forward secrecy)
- Prekeys (so you can message someone who's offline)
- Many other clever tricks

We'll learn the **basics** using **RSA encryption** — the fundamental concept is the same:
**only the recipient can read the message**. 🎯

---

## 🔑 Asymmetric Encryption 101

The magic behind E2E encryption is **asymmetric encryption** (also called "public-key cryptography").

It uses **two keys** that are mathematically linked:

| Key | Analogy | Who has it? | What it does |
|-----|---------|-------------|--------------|
| **Public Key** 🔓 | An open padlock | Everyone (it's public!) | Locks (encrypts) messages |
| **Private Key** 🔐 | The key to the padlock | Only the owner (keep it SECRET!) | Unlocks (decrypts) messages |

### The Padlock Analogy 🔒

Think of it this way:

1. **Bob creates a padlock and a key** (generates a key pair)
2. **Bob gives copies of his OPEN padlock to everyone** (shares his public key)
3. **Bob keeps the key safe in his pocket** (keeps his private key secret)
4. **Alice puts her message in a box and snaps Bob's padlock shut** (encrypts with Bob's public key)
5. **Only Bob can open it with his key** (decrypts with his private key)

The beauty: **anyone can lock the box, but only Bob can unlock it!**

Let's see this in action with real code! 👇

In [ ]:
# === 🔑 Generate an RSA Key Pair ===
# This is like creating a padlock (public key) and its key (private key)

# Generate a private key (this also contains the public key)
private_key = rsa.generate_private_key(
    public_exponent=65537,  # Standard value, don't worry about this
    key_size=2048,          # 2048 bits = very strong encryption
)

# Extract the public key from the private key
public_key = private_key.public_key()

print("🔐 Key pair generated!")
print(f"   Key size: 2048 bits (that's VERY hard to crack)")
print()

# Let's see what the public key looks like (PEM format)
public_key_pem = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo,
)
print("🔓 PUBLIC KEY (safe to share with everyone):")
print(public_key_pem.decode())

# And the private key (PEM format)
private_key_pem = private_key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),  # No password for demo
)
print("🔐 PRIVATE KEY (keep this SECRET — never share it!):")
print(private_key_pem.decode()[:200] + "...")
print()
print("⚠️  In a real app, the private key NEVER leaves your device!")

---

## ✉️ Encrypting a Message

Now let's actually encrypt a message! We'll use **RSA-OAEP** padding, which is
a secure way to use RSA encryption.

Think of it as: **locking a message in a box using the recipient's padlock**.

In [ ]:
# === ✉️ Encrypt a Message ===

# Our secret message
plaintext_message = "Hey Bob! The surprise party is at 7pm 🎉"

print(f"📝 Original message: {plaintext_message}")
print()

# Encrypt the message using the PUBLIC key
# (Anyone who has the public key can do this)
ciphertext = public_key.encrypt(
    plaintext_message.encode("utf-8"),  # Convert string to bytes
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),  # Mask generation function
        algorithm=hashes.SHA256(),                      # Hash algorithm
        label=None,
    ),
)

# Convert to base64 so we can display it as text
ciphertext_b64 = base64.b64encode(ciphertext).decode("utf-8")

print("🔒 Encrypted message (ciphertext):")
print(f"   {ciphertext_b64[:80]}...")
print()
print(f"📏 Original message length: {len(plaintext_message)} characters")
print(f"📏 Encrypted message length: {len(ciphertext_b64)} characters")
print()

# Let's try to "read" the encrypted message...
print("🤔 Can we read the encrypted data as text?")
try:
    gibberish = ciphertext.decode("utf-8")
except UnicodeDecodeError:
    print("   ❌ Nope! It's just random-looking bytes — complete gibberish!")
    print(f"   Raw bytes (first 50): {ciphertext[:50]}")
    print()
    print("   🎯 This is exactly what we want — nobody can read it without the private key!")

---

## 🔓 Decrypting a Message

Now let's be Bob and **unlock** the message using the **private key**.

Remember: only the person with the private key can do this!

In [ ]:
# === 🔓 Decrypt the Message ===

# Use the PRIVATE key to decrypt
# (Only the owner of the private key can do this!)
decrypted_bytes = private_key.decrypt(
    ciphertext,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)

# Convert bytes back to string
decrypted_message = decrypted_bytes.decode("utf-8")

print("🔓 Decrypted message:")
print(f"   {decrypted_message}")
print()

# Verify it matches the original -- assert, so a broken round-trip stops the
# notebook instead of printing a sad face and carrying on.
assert decrypted_message == plaintext_message, (
    f"decrypt(encrypt(m)) must return m exactly; got {decrypted_message!r}"
)
print("✅ Perfect! The decrypted message matches the original!")
print()
print("🎯 Key takeaway:")
print("   • Anyone can ENCRYPT with the public key (lock the box)")
print("   • Only the private key holder can DECRYPT (open the box)")

---

## 🤝 Key Exchange

Before Alice and Bob can send encrypted messages, they need to **exchange public keys**.

Here's how it works:

```
1. Alice generates her key pair (public + private)
2. Bob generates his key pair (public + private)
3. Alice uploads her PUBLIC key to the server
4. Bob uploads his PUBLIC key to the server
5. When Alice wants to message Bob, she downloads Bob's public key from the server
6. Alice encrypts her message with Bob's public key
7. Only Bob can decrypt it with his private key!
```

**Important**: The server stores public keys, but it **NEVER** sees private keys.
Private keys stay on the user's device at all times.

Let's set this up for Alice and Bob! 👇

In [ ]:
# === 🤝 Key Exchange: Generate Key Pairs for Alice and Bob ===

# --- Alice generates her keys ---
alice_private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
)
alice_public_key = alice_private_key.public_key()

# --- Bob generates his keys ---
bob_private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
)
bob_public_key = bob_private_key.public_key()

print("🔑 Key pairs generated!")
print("   👩 Alice: ✅ public key + ✅ private key")
print("   👨 Bob:   ✅ public key + ✅ private key")
print()

# Convert public keys to PEM format (text) for storage
alice_public_pem = alice_public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo,
).decode("utf-8")

bob_public_pem = bob_public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo,
).decode("utf-8")

print("📤 Public keys ready to upload to the server!")
print(f"   Alice's public key: {alice_public_pem[:60]}...")
print(f"   Bob's public key:   {bob_public_pem[:60]}...")

In [ ]:
# === 📤 Store Public Keys in the Database ===
# In a real app, this happens when a user registers or sets up encryption

# Upload Alice's public key to the server
run_query(
    "UPDATE users SET public_key = %s WHERE username = 'alice';",
    (alice_public_pem,),
    fetch=False,
)

# Upload Bob's public key to the server
run_query(
    "UPDATE users SET public_key = %s WHERE username = 'bob';",
    (bob_public_pem,),
    fetch=False,
)

print("✅ Public keys stored in the database!")
print()

# Let's verify by querying the database
results = run_query("""
    SELECT username, 
           CASE WHEN public_key IS NOT NULL 
                THEN '🔓 Has public key' 
                ELSE '❌ No public key' 
           END AS key_status,
           LEFT(public_key, 40) AS key_preview
    FROM users 
    ORDER BY id;
""")

print("📋 Users and their encryption status:")
print("-" * 65)
for r in results:
    preview = r['key_preview'] or 'N/A'
    print(f"   {r['username']:10} {r['key_status']:20} {preview}...")

print()
print("🔐 Remember: Private keys stay on Alice's and Bob's devices!")
print("   The server ONLY has public keys (padlocks, not keys).")

---

## 📨 Sending an Encrypted Message (Full Flow)

Now let's see the **complete flow** of sending an encrypted message,
just like WhatsApp does it:

```
1. Alice wants to send "Hello Bob!" to Bob
2. Alice fetches Bob's public key from the server
3. Alice encrypts the message with Bob's public key
4. Alice sends the encrypted message to the server
5. The server stores the encrypted message (it can't read it!)
6. Bob downloads the encrypted message from the server
7. Bob decrypts it with his private key
8. Bob reads: "Hello Bob!" 🎉
```

In [ ]:
# === 📨 Full Encrypted Message Flow ===

# --- Helper functions ---

def encrypt_message(plaintext, recipient_public_key):
    """Encrypt a message using the recipient's public key."""
    ciphertext = recipient_public_key.encrypt(
        plaintext.encode("utf-8"),
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None,
        ),
    )
    return base64.b64encode(ciphertext).decode("utf-8")


def decrypt_message(ciphertext_b64, recipient_private_key):
    """Decrypt a message using the recipient's private key."""
    ciphertext = base64.b64decode(ciphertext_b64)
    plaintext_bytes = recipient_private_key.decrypt(
        ciphertext,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None,
        ),
    )
    return plaintext_bytes.decode("utf-8")


def load_public_key_from_pem(pem_string):
    """Load a public key from its PEM text representation."""
    return serialization.load_pem_public_key(pem_string.encode("utf-8"))


print("✅ Helper functions ready!")

In [ ]:
# === 📨 Step-by-Step: Alice Sends an Encrypted Message to Bob ===

secret_message = "Hey Bob! Meet me at the coffee shop at 3pm ☕"

print("=" * 60)
print("📨 ALICE SENDING AN ENCRYPTED MESSAGE TO BOB")
print("=" * 60)
print()

# --- Step 1: Alice's device fetches Bob's public key from the server ---
print("📥 Step 1: Alice fetches Bob's public key from the server")
result = run_query(
    "SELECT public_key FROM users WHERE username = 'bob';"
)
bob_public_pem_from_db = result[0]["public_key"]
bob_pub_key_loaded = load_public_key_from_pem(bob_public_pem_from_db)
print(f"   ✅ Got Bob's public key from database")
print()

# --- Step 2: Alice encrypts the message on her device ---
print("🔒 Step 2: Alice encrypts the message with Bob's public key")
print(f"   📝 Original: {secret_message}")
encrypted_content = encrypt_message(secret_message, bob_pub_key_loaded)
print(f"   🔒 Encrypted: {encrypted_content[:60]}...")
print()

# --- Step 3: Send the encrypted message to the server (store in DB) ---
print("📤 Step 3: Alice sends the encrypted message to the server")

# Find the chat between Alice and Bob
chat = run_query("""
    SELECT c.id 
    FROM chats c
    JOIN chat_participants cp1 ON c.id = cp1.chat_id
    JOIN chat_participants cp2 ON c.id = cp2.chat_id
    JOIN users u1 ON cp1.user_id = u1.id
    JOIN users u2 ON cp2.user_id = u2.id
    WHERE u1.username = 'alice' AND u2.username = 'bob' AND c.is_group = FALSE
    LIMIT 1;
""")
chat_id = chat[0]["id"]

alice_user = run_query("SELECT id FROM users WHERE username = 'alice';")
alice_id = alice_user[0]["id"]

# Remember where the counter was, so the cleanup cell can rewind it.
seq_before = run_query(
    "SELECT last_sequence FROM chat_sequences WHERE chat_id = %s;", (chat_id,)
)[0]["last_sequence"]

# Allocate the sequence number ATOMICALLY -- one UPDATE ... RETURNING, exactly
# as store_message() does in server/chat_server.py. Reading the counter and
# then bumping it in a second statement (an earlier version of this cell) is a
# read-modify-write race: two concurrent senders both read N and both write N+1.
seq = run_query("""
    UPDATE chat_sequences SET last_sequence = last_sequence + 1
    WHERE chat_id = %s
    RETURNING last_sequence;
""", (chat_id,))[0]["last_sequence"]

# Store the message. `content` holds a placeholder -- the real text only ever
# exists as ciphertext in encrypted_content.
encrypted_message_id = run_query("""
    INSERT INTO messages (chat_id, sender_id, content, encrypted_content, sequence_number)
    VALUES (%s, %s, '[encrypted]', %s, %s)
    RETURNING id;
""", (chat_id, alice_id, encrypted_content, seq))[0]["id"]

assert seq == seq_before + 1, f"sequence should advance by exactly 1, {seq_before} -> {seq}"
print(f"   ✅ Encrypted message stored (chat_id={chat_id}, id={encrypted_message_id}, seq={seq})")
print()

# --- Step 4: Show what the server sees ---
print("👁️ Step 4: What does the SERVER see in the database?")
server_view = run_query("""
    SELECT content, LEFT(encrypted_content, 60) AS encrypted_preview
    FROM messages WHERE id = %s;
""", (encrypted_message_id,))
print(f"   content:           {server_view[0]['content']}")
print(f"   encrypted_content: {server_view[0]['encrypted_preview']}...")
print(f"   🔐 The server only sees gibberish! It CANNOT read the message.")
print()

# --- Step 5: Bob receives and decrypts ---
print("📥 Step 5: Bob downloads and decrypts the message")
encrypted_from_db = run_query(
    "SELECT encrypted_content FROM messages WHERE id = %s;", (encrypted_message_id,)
)
encrypted_blob = encrypted_from_db[0]["encrypted_content"]

# Bob decrypts with his PRIVATE key (which only Bob has!)
decrypted = decrypt_message(encrypted_blob, bob_private_key)
print(f"   🔓 Decrypted: {decrypted}")
print()

assert decrypted == secret_message, (
    f"Bob must recover Alice's exact plaintext, got {decrypted!r}"
)
assert server_view[0]["content"] == "[encrypted]", (
    "the plaintext must never be written to the database -- the server row "
    f"contains {server_view[0]['content']!r}"
)
assert secret_message not in (encrypted_blob or ""), (
    "the plaintext leaked into the stored ciphertext"
)
print("🎉 SUCCESS! The message was delivered securely!")
print("   ✅ Alice encrypted it — only Bob could read it")
print("   ✅ The server never saw the plaintext")
print("   ✅ This is End-to-End Encryption!")

---

## 🕵️ Why the Server Can't Read Your Messages

This is the **whole point** of End-to-End Encryption. Let's prove it!

In [ ]:
# === 🕵️ Proving the Server Can't Read Messages ===

print("=" * 60)
print("🕵️ LET'S BE THE SERVER ADMIN AND TRY TO READ MESSAGES")
print("=" * 60)
print()

# --- As the server admin, query ALL messages ---
print("📋 Server admin queries the messages table:")
print()
all_messages = run_query("""
    SELECT m.id, u.username AS sender, m.content, 
           LEFT(m.encrypted_content, 50) AS encrypted_preview
    FROM messages m
    JOIN users u ON m.sender_id = u.id
    ORDER BY m.server_timestamp DESC
    LIMIT 5;
""")

for msg in all_messages:
    enc = msg['encrypted_preview'] or 'NULL'
    print(f"   ID: {msg['id']} | From: {msg['sender']:10} | Content: {msg['content'][:30]:30} | Encrypted: {enc[:30]}...")

print()
print("🤔 The admin can see unencrypted messages (the old ones),")
print("   but the E2E encrypted message just shows '[encrypted]' and gibberish!")
print()

In [ ]:
# === 🔓 Can the Server Decrypt Without Bob's Private Key? ===

print("🕵️ Server admin tries to decrypt the message...")
print()

# The server admin has access to the database. Can they decrypt?
encrypted_msg = run_query("""
    SELECT encrypted_content FROM messages
    WHERE encrypted_content IS NOT NULL
    ORDER BY server_timestamp DESC LIMIT 1;
""")
encrypted_blob = encrypted_msg[0]["encrypted_content"]

# The server could try generating a random key...
print("🔑 Attempt 1: Generate a random key and try to decrypt")
fake_private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
)

wrong_key_worked = False
try:
    ciphertext_bytes = base64.b64decode(encrypted_blob)
    fake_private_key.decrypt(
        ciphertext_bytes,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None,
        ),
    )
    wrong_key_worked = True
except Exception as e:
    print(f"   ❌ FAILED: {type(e).__name__}")
    print(f"   The wrong key cannot decrypt the message!")

assert not wrong_key_worked, (
    "an unrelated private key decrypted the ciphertext -- this notebook's "
    "entire claim is false if that ever happens"
)

print()

# Attack the maths instead of the key space.
print("🔑 Attempt 2: Break the maths?")
print("   Careful with the numbers here -- a common myth is that RSA-2048 gives")
print("   you 2048 bits of security. It does not. Nobody enumerates RSA keys;")
print("   you factor the 2048-bit modulus n = p * q and derive the private key.")
print("   The best known algorithm (the General Number Field Sieve) puts RSA-2048")
print("   at roughly 112-bit security -- about 2^112 operations, not 2^2048.")
print()
print("   2^112 is still ~5 x 10^33 operations. At a trillion tries per second")
print("   that is ~10^14 years, or ~10,000x the age of the universe. 🌌")
print("   But the gap matters: it is why 1024-bit RSA is dead and 2048 is the")
print("   floor, while a 256-bit symmetric key (2^256) is overkill by comparison.")
print()

print("=" * 60)
print("🔐 CONCLUSION: The server admin CANNOT read your messages!")
print("   This is the power of End-to-End Encryption.")
print("   Even WhatsApp/Meta themselves cannot read your messages.")
print("=" * 60)

---

## 👥 Group Encryption Challenge

So far, we've encrypted messages for **one recipient**. But what about **group chats**?

### The Problem

In a group chat with 3 people (Alice, Bob, Charlie), each person has a **different** public key.
If Alice sends a message, she needs EVERYONE to be able to decrypt it.

**Naive approach**: Encrypt the message separately for each recipient.

```
Alice's message: "Hello group!"

→ Encrypt with Bob's public key      → Send to Bob
→ Encrypt with Charlie's public key   → Send to Charlie
```

This works, but it means **one message becomes N-1 messages** (one per group
member other than the sender). In a group of 256 people that's 255 encryptions —
and worse, RSA-2048 with OAEP can only wrap about **190 bytes**, so the naive
scheme cannot send a long message *at all*. 😰

Let's see this in action and discuss a better approach. 👇

In [ ]:
# === 👥 Group Encryption: Encrypt for Multiple Recipients ===

# Generate key pairs for 3 group members
group_members = {}
for name in ["alice", "bob", "charlie"]:
    priv = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    group_members[name] = {
        "private_key": priv,
        "public_key": priv.public_key(),
    }

print("🔑 Generated key pairs for group members:")
for name in group_members:
    print(f"   👤 {name}: ✅ public key + ✅ private key")
print()

# Alice sends a message to the group
group_message = "Hello everyone! Study session at 5pm today 📚"
print(f"📝 Alice's message: {group_message}")
print()

# Encrypt the same message for each recipient
print("🔒 Encrypting for each group member:")
encrypted_copies = {}
for name, keys in group_members.items():
    if name == "alice":  # Alice is the sender, she already knows the message
        continue
    encrypted = encrypt_message(group_message, keys["public_key"])
    encrypted_copies[name] = encrypted
    print(f"   🔒 For {name}: {encrypted[:40]}...")

print()
print(f"📊 Results:")
print(f"   Original message: 1 copy ({len(group_message)} chars)")
print(f"   Encrypted copies: {len(encrypted_copies)} copies")
print(f"   Each encrypted copy: ~{len(list(encrypted_copies.values())[0])} chars")
print()

# Each recipient can decrypt their own copy
print("🔓 Each member decrypts their copy:")
for name, encrypted in encrypted_copies.items():
    decrypted = decrypt_message(encrypted, group_members[name]["private_key"])
    print(f"   👤 {name} reads: {decrypted}")
print()

# But they can't read each other's copies!
assert len(encrypted_copies) == len(group_members) - 1, (
    f"one message must become {len(group_members) - 1} ciphertexts (everyone "
    f"but the sender), got {len(encrypted_copies)}"
)
assert len(set(encrypted_copies.values())) == len(encrypted_copies), (
    "each recipient's ciphertext must be distinct -- identical blobs would mean "
    "we encrypted with the same key twice"
)

print("🚫 Can Bob decrypt Charlie's copy?")
bob_read_charlies_copy = False
try:
    decrypt_message(encrypted_copies["charlie"], group_members["bob"]["private_key"])
    bob_read_charlies_copy = True
except Exception:
    print("   ❌ Nope! Each copy is encrypted with a DIFFERENT key.")

assert not bob_read_charlies_copy, (
    "Bob decrypted a ciphertext addressed to Charlie -- per-recipient encryption "
    "is not actually per-recipient"
)

In [ ]:
# === 📦 Envelope Encryption: The Smarter Approach ===
# This is what real messaging apps use for groups.

import os

from cryptography.hazmat.primitives.ciphers.aead import AESGCM

print("=" * 60)
print("📦 ENVELOPE ENCRYPTION (How Real Group Chats Work)")
print("=" * 60)
print()
print("Instead of encrypting the message N times, we:")
print("  1. Generate a random 'group key' (symmetric, AES-256)")
print("  2. Encrypt the MESSAGE once with the group key   (fast, any size)")
print("  3. Encrypt the GROUP KEY for each member         (RSA, 32 tiny bytes)")
print()

# Step 1: a fresh random symmetric key + nonce for THIS message.
group_key = os.urandom(32)   # 256-bit AES key
nonce = os.urandom(12)       # 96-bit nonce -- never reuse one with the same key
aesgcm = AESGCM(group_key)
print(f"🔑 Random group key: {base64.b64encode(group_key).decode()[:30]}...")
print()

# Step 2: encrypt the message ONCE.
#
# We use AES-GCM, not AES-CBC. GCM is *authenticated*: the ciphertext carries a
# tag, so decryption fails loudly if a single bit was flipped. Plain CBC gives
# you confidentiality with NO integrity -- the server (or anyone on the path)
# could tamper with the ciphertext and the recipient would never know. That is
# not a nitpick: unauthenticated CBC is the root of a long line of real
# vulnerabilities (padding oracles). GCM also does its own padding, so the
# hand-rolled pad-to-16-bytes step disappears.
encrypted_message = aesgcm.encrypt(nonce, group_message.encode("utf-8"), None)
print(f"🔒 Message encrypted once with AES-256-GCM: "
      f"{base64.b64encode(encrypted_message).decode()[:40]}...")
print(f"   ({len(group_message.encode())} plaintext bytes -> {len(encrypted_message)} bytes, "
      f"including the 16-byte auth tag)")
print()

# Step 3: seal the group key for each member with their RSA public key.
sealed_keys = {}
print("🔑 Sealing the group key for each member:")
for name, keys in group_members.items():
    if name == "alice":            # the sender already has the key
        continue
    sealed_keys[name] = keys["public_key"].encrypt(
        group_key + nonce,          # 32 + 12 = 44 bytes, comfortably under RSA's limit
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None,
        ),
    )
    print(f"   🔐 for {name}: {len(sealed_keys[name])} bytes")
print()

# --- Prove it round-trips for every recipient ---
for name, sealed in sealed_keys.items():
    opened = group_members[name]["private_key"].decrypt(
        sealed,
        padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()),
                     algorithm=hashes.SHA256(), label=None),
    )
    their_key, their_nonce = opened[:32], opened[32:]
    plaintext = AESGCM(their_key).decrypt(their_nonce, encrypted_message, None)
    assert plaintext.decode("utf-8") == group_message, (
        f"{name} could not recover the group message from the shared envelope"
    )
    print(f"   🔓 {name} opened the envelope and read: {plaintext.decode()[:45]}")

# --- Prove the auth tag actually catches tampering ---
tampered = bytearray(encrypted_message)
tampered[0] ^= 0x01                      # flip one bit of ciphertext
tamper_went_unnoticed = False
try:
    aesgcm.decrypt(nonce, bytes(tampered), None)
    tamper_went_unnoticed = True
except Exception as e:
    print(f"\n🛡️  Flipped one bit -> decryption REJECTED it ({type(e).__name__}).")
assert not tamper_went_unnoticed, (
    "a one-bit change to the ciphertext decrypted successfully -- this cipher "
    "is not authenticated, and the whole point of using GCM was integrity"
)

# --- Cost comparison, computed rather than asserted by hand ---
members = 100
naive_rsa_ops = members - 1                        # one RSA encrypt per recipient
envelope_rsa_ops = members - 1                     # ...still one per recipient!
naive_bytes = (members - 1) * len(group_message.encode())
envelope_bytes = len(encrypted_message) + (members - 1) * 44

print(f"\n📊 For a group of {members}, one message:")
print(f"   Naive:    {naive_rsa_ops} RSA encryptions over "
      f"{naive_bytes:,} bytes of plaintext")
print(f"   Envelope: {envelope_rsa_ops} RSA encryptions over "
      f"{envelope_bytes:,} bytes (a fixed 44 per member, plus one AES pass)")
print()
print("💡 Note what does NOT change: you still do one RSA operation per member.")
print("   Envelope encryption doesn't dodge the O(N) fan-out -- it makes each")
print("   unit tiny and constant-sized, and it lets the MESSAGE be any length")
print("   (RSA-2048 + OAEP can only wrap ~190 bytes, so the naive scheme cannot")
print("   send a long message at all). That size ceiling, not speed, is the")
print("   reason every real system reaches for a hybrid scheme.")
print()
print("   The Signal Protocol goes further still: a per-group 'sender key' is")
print("   distributed once, then ratcheted -- so the per-message cost drops to")
print("   one symmetric encryption for the whole group.")

---

## ⚠️ Limitations & Real-World Considerations

What we've built is a **simplified demo**. Real-world E2E encryption (like WhatsApp's) is
much more sophisticated. Here are some important differences:

### 🔄 Signal Protocol vs Basic RSA

| Feature | Our Demo (RSA) | WhatsApp (Signal Protocol) |
|---------|---------------|---------------------------|
| Key type | Static RSA keys | Rotating ephemeral keys |
| Forward secrecy | ❌ No | ✅ Yes (Double Ratchet) |
| Key exchange | Manual | Automatic (X3DH) |
| Message size | Limited by key size | Unlimited (hybrid encryption) |
| Performance | Slow for large data | Fast (AES + key exchange) |

### 🔐 Forward Secrecy

With our demo, if someone steals Bob's private key, they can decrypt **ALL** past messages.

With **Forward Secrecy** (used in Signal Protocol), keys change with every message.
So even if a key is compromised, only that ONE message is exposed — not the entire history.

Think of it like using a **new padlock for every single message**! 🔒🔒🔒

### ✅ Key Verification (QR Codes)

How do you know the public key on the server actually belongs to Bob?
What if an attacker replaced it with their own key? (This is called a "man-in-the-middle" attack.)

WhatsApp solves this with **QR code verification**:
- You meet Bob in person
- You scan each other's QR codes
- This confirms you both have each other's real public keys

### 📱 New Device Setup

What happens when you get a new phone?
- Your private key was on the old phone
- You generate a **new key pair** on the new phone
- Everyone gets a notification: "Bob's security code changed"
- Old messages encrypted with the old key **cannot** be decrypted on the new phone
  (unless you transfer the key or have a backup)

### 🌍 Other Real-World Challenges

- **Multi-device support**: How to have the same keys on phone + laptop?
- **Key backup**: What if you lose your phone? (WhatsApp offers encrypted cloud backups)
- **Metadata**: E2E protects message *content*, but the server still sees *who* messages *whom* and *when*
- **Group key rotation**: When someone leaves a group, all keys must change

> 📖 **Want to learn more?** Read the [Signal Protocol specification](https://signal.org/docs/)
> or the [WhatsApp Security Whitepaper](https://www.whatsapp.com/security/).

---

## 🧹 Cleanup

Let's reset the database to its original state.

In [ ]:
# === 🧹 Cleanup: Reset to Original State ===

# Remove public keys from users table
run_query("UPDATE users SET public_key = NULL;", fetch=False)
print("✅ Reset public_key to NULL for all users")

# Delete any encrypted test messages we created
run_query(
    "DELETE FROM messages WHERE encrypted_content IS NOT NULL;",
    fetch=False,
)
print("✅ Deleted test encrypted messages")

# Rewind the sequence counter too. Deleting the message without this leaves the
# counter permanently ahead of the highest stored sequence -- harmless in a demo,
# but it means "back to its original state" below would be a lie, and re-running
# the notebook would drift further every time.
run_query("UPDATE chat_sequences SET last_sequence = %s WHERE chat_id = %s;",
          (seq_before, chat_id), fetch=False)
print(f"✅ Reset chat {chat_id}'s sequence counter to {seq_before}")

# Verify cleanup
remaining = run_query("""
    SELECT COUNT(*) AS encrypted_count 
    FROM messages 
    WHERE encrypted_content IS NOT NULL;
""")
keys = run_query("""
    SELECT COUNT(*) AS key_count 
    FROM users 
    WHERE public_key IS NOT NULL;
""")
print(f"\n📋 Verification:")
print(f"   Encrypted messages remaining: {remaining[0]['encrypted_count']}")
print(f"   Users with public keys: {keys[0]['key_count']}")
assert remaining[0]["encrypted_count"] == 0 and keys[0]["key_count"] == 0, (
    f"cleanup left {remaining[0]['encrypted_count']} encrypted message(s) and "
    f"{keys[0]['key_count']} stored public key(s) behind"
)
counter = run_query("SELECT last_sequence FROM chat_sequences WHERE chat_id = %s;",
                    (chat_id,))[0]["last_sequence"]
highest = run_query("SELECT COALESCE(MAX(sequence_number), 0) AS m FROM messages "
                    "WHERE chat_id = %s;", (chat_id,))[0]["m"]
assert counter == highest, (
    f"chat {chat_id}: counter is {counter} but the highest stored sequence is "
    f"{highest} -- the counter drifted"
)
print(f"   Chat {chat_id} sequence counter: {counter} (matches stored messages)")

print("\n🧹 Database cleaned up!")

---

## 📝 Summary

### Key Takeaways from This Notebook

| Concept | What You Learned |
|---------|------------------|
| **E2E Encryption** | Messages are encrypted on the sender's device and decrypted on the recipient's device. The server NEVER sees plaintext. |
| **Public Key** 🔓 | Like an open padlock — anyone can use it to encrypt (lock) a message for you |
| **Private Key** 🔐 | Like the key to the padlock — only YOU can use it to decrypt (unlock) messages |
| **Key Exchange** | Users upload public keys to the server; private keys NEVER leave the device |
| **Group Encryption** | Naive: encrypt per-recipient. Smart: envelope encryption (encrypt once + encrypt the key per-recipient) |
| **Real-World** | Signal Protocol adds forward secrecy, key rotation, and many other improvements over basic RSA |

### 🔐 The Golden Rule of E2E

> **"Even if someone breaks into the server, your messages are still safe."**

This is what makes End-to-End Encryption so powerful. The server is just a messenger —
it carries locked boxes but never has the keys to open them.

---

## 🎉 Series Complete!

Congratulations! You've completed all 4 notebooks in the WhatsApp System Design Lab!

Here's what you've learned across the series:

| Notebook | What You Built |
|----------|----------------|
| **1. Message Delivery & Storage** | WebSocket messaging, PostgreSQL persistence, offline delivery |
| **2. Read Receipts & Presence** | Online/offline status, message read tracking, real-time updates |
| **3. Group Messaging** | Group chats, fan-out delivery, admin controls |
| **4. End-to-End Encryption** | RSA encryption, key exchange, group encryption, server-proof security |

### 🚀 Where to Go Next

- 📖 Read the [Signal Protocol docs](https://signal.org/docs/) to learn production E2E encryption
- 🏗️ Try implementing forward secrecy with the Double Ratchet Algorithm
- 🔍 Explore the [WhatsApp Security Whitepaper](https://www.whatsapp.com/security/)
- 💡 Think about: How would you handle E2E encryption with message search?

**Happy building! 🛠️**